In [1]:
%pip install -U ripser persim

Note: you may need to restart the kernel to use updated packages.


In [1]:
import os
import gc
import random
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from dataclasses import dataclass
from typing import List, Dict, Optional
from tqdm import tqdm
from transformers import AutoTokenizer, AutoModelForCausalLM
from sklearn.neighbors import NearestNeighbors
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from datasets import load_dataset

# --- TDA Support ---
try:
    from ripser import ripser
    HAS_RIPSER = True
except ImportError:
    HAS_RIPSER = False
    print("⚠️ WARNING: 'ripser' not found. Topology metrics (Betti) will be NaN.")

# ============================================================
# 1. Configuration
# ============================================================

@dataclass
class ModelCfg:
    model_id: str
    short_name: str
    dtype: torch.dtype = torch.bfloat16
    trust_remote_code: bool = True
    attn_implementation: str = "sdpa"

@dataclass
class ExpConfig:
    # Data Settings
    n_samples: int = 500       
    batch_size: int = 8
    max_length: int = 128
    seed: int = 42
    
    # Dataset Paths
    local_csv_impossible: str = "impossibleQ.csv"
    local_csv_factual: str = "factual.csv"

    # Analysis Toggles
    layers_to_scan: Optional[List[int]] = None 
    compute_geometry: bool = True  # LID, Iso, Ent
    compute_topology: bool = True  # Betti
    compute_mechanics: bool = True # Fisher, Hess, Jac, Grad, Vocab
    
    # Hyperparams
    lid_k: int = 30
    advanced_subset_size: int = 48

def set_seed(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

# ============================================================
# 2. Data Loading
# ============================================================

def get_real_datasets(cfg: ExpConfig) -> Dict[str, List[str]]:
    set_seed(cfg.seed)
    datasets_out = {}
    print(f"Loading datasets (Target: {cfg.n_samples} per bucket)...")

    # Helper function to read local CSVs
    def load_local_csv(path, label):
        prompts = []
        if os.path.exists(path):
            try:
                df = pd.read_csv(path)
                # Find a column that looks like 'question' or 'text'
                col = next((c for c in df.columns if 'question' in c.lower() or 'text' in c.lower()), None)
                if col:
                    raw = df[col].astype(str).tolist()
                    # Sample if we have too many, otherwise take all
                    if len(raw) > cfg.n_samples:
                        raw = list(np.random.choice(raw, cfg.n_samples, replace=False))
                    prompts = [f"Question: {t}\nAnswer:" for t in raw]
                    print(f"    > Loaded {len(prompts)} from {path} ({label})")
                else:
                    print(f"    ! CSV {path} found but no 'question'/'text' column.")
            except Exception as e:
                print(f"    ! Error reading {path}: {e}")
        else:
            print(f"    ! File not found: {path}")
        return prompts

    # 1. Factual (From local factual.csv)
    datasets_out['1_Factual'] = load_local_csv(cfg.local_csv_factual, "Factual")

    # 2. Impossible (From local impossibleQ.csv)
    datasets_out['2_Impossible'] = load_local_csv(cfg.local_csv_impossible, "Impossible")

    # 3. Hallucinations (Mix of TruthfulQA + Low-Pop PopQA)
    try:
        print("    > Building Mixed Hallucination Dataset...")
        mixed_pool = []

        # A. Load TruthfulQA
        try:
            ds_t = load_dataset("truthful_qa", "generation", split="validation")
            t_questions = [row['question'] for row in ds_t]
            mixed_pool.extend(t_questions)
            print(f"      - Added {len(t_questions)} from TruthfulQA")
        except Exception as e:
            print(f"      ! Error loading TruthfulQA: {e}")

        # B. Load PopQA (Low Popularity)
        try:
            ds_p = load_dataset("akariasai/PopQA", split="test")
            df_pop = ds_p.to_pandas()
            # Sort ascending (Low s_pop = obscure) and take bottom 50%
            df_pop = df_pop.sort_values(by="s_pop", ascending=True)
            # Take enough to roughly match TruthfulQA size or fill the pool
            p_questions = df_pop['question'].tolist()[:1000] 
            mixed_pool.extend(p_questions)
            print(f"      - Added {len(p_questions)} from PopQA (Low Pop)")
        except Exception as e:
            print(f"      ! Error loading PopQA: {e}")

        # C. Shuffle and Trim
        if mixed_pool:
            random.shuffle(mixed_pool)
            final_hallucination = mixed_pool[:cfg.n_samples]
            datasets_out['3_Hallucination'] = [f"Question: {q}\nAnswer:" for q in final_hallucination]
            print(f"    > Final Hallucination size: {len(datasets_out['3_Hallucination'])}")
        else:
            print("    ! Failed to build Hallucination dataset (sources empty).")
            datasets_out['3_Hallucination'] = []

    except Exception as e:
        print(f"    ! General error building Hallucinations: {e}")

    return datasets_out
    
# ============================================================
# 3. Metric Engine (Updated with Boundary Norm & Direct Proj)
# ============================================================

class MetricEngine:
    @staticmethod
    def get_last_token_reps(hidden_states, attention_mask):
        last_indices = attention_mask.sum(dim=1) - 1
        last_indices = last_indices.clamp(min=0)
        batch_size = hidden_states.shape[0]
        return hidden_states[torch.arange(batch_size, device=hidden_states.device), last_indices, :]

    @staticmethod
    def compute_lid(X, k=20):
        if len(X) <= k: return np.nan
        X = np.array(X, dtype=np.float64)
        try:
            nn = NearestNeighbors(n_neighbors=k+1, n_jobs=-1).fit(X)
            dists, _ = nn.kneighbors(X)
            dists = dists[:, 1:]
            r_max = dists[:, -1]
            mask = r_max > 1e-9
            if not np.any(mask): return 0.0
            dists, r_max = dists[mask], r_max[mask, None]
            lids = -k / np.sum(np.log(dists / r_max + 1e-10), axis=1)
            return np.mean(lids)
        except:
            return np.nan

    @staticmethod
    def compute_isotropy(X):
        if len(X) < 5: return np.nan
        try:
            pca = PCA(n_components=2).fit(X)
            ev = pca.explained_variance_ratio_
            return ev[1] / (ev[0] + 1e-9)
        except:
            return np.nan
    
    @staticmethod
    def compute_entropy(X):
        X_c = X - X.mean(axis=0)
        try:
            _, S, _ = np.linalg.svd(X_c, full_matrices=False)
            if np.sum(S) == 0: return 0.0
            S_norm = S / np.sum(S)
            entropy = -np.sum(S_norm * np.log(S_norm + 1e-10))
            return np.exp(entropy)
        except:
            return np.nan

    @staticmethod
    def compute_boundary_vector(X_factual, X_impossible):
        """
        UPDATED: Returns (unit_vector, raw_norm).
        Tracks absolute distance between class means.
        """
        mu_f = X_factual.astype(np.float64).mean(axis=0)
        mu_i = X_impossible.astype(np.float64).mean(axis=0)
        vec = mu_i - mu_f
        raw_norm = np.linalg.norm(vec)
        return vec / (raw_norm + 1e-9), raw_norm

    @staticmethod
    def compute_boundary_topology(X, boundary_vec, quantile=0.40):
        """
        FIXED: Uses Cosine Distance + pairwise_distances to overcome 
        the curse of dimensionality in high-dim Euclidean space.
        """
        # Need this import
        from sklearn.metrics import pairwise_distances
        
        if not HAS_RIPSER or len(X) < 10: return np.nan, np.nan
        try:
            # 1. Filter points based on projection (same as before)
            X = X.astype(np.float64)
            boundary_vec = boundary_vec.astype(np.float64)
            
            proj = X @ boundary_vec
            center = np.mean(proj)
            dists_proj = np.abs(proj - center)
            thresh_proj = np.quantile(dists_proj, quantile)
            X_sub = X[dists_proj <= thresh_proj]
            
            # Subsample if too large (TDA is expensive)
            if len(X_sub) > 200:
                indices = np.random.choice(len(X_sub), 200, replace=False)
                X_sub = X_sub[indices]
            
            # --- FIX STARTS HERE ---
            
            # 2. Compute Distance Matrix (Cosine)
            # Cosine Distance = 1 - Cosine Similarity. Range [0, 2].
            # 0.0 = Identical, 1.0 = Orthogonal, 2.0 = Opposite
            d_matrix = pairwise_distances(X_sub, metric='cosine')
            
            # 3. Run Ripser on the Distance Matrix
            # maxdim=1 computes H0 (Clusters) and H1 (Loops)
            res = ripser(d_matrix, distance_matrix=True, maxdim=1)
            diagrams = res['dgms']
            
            # 4. Define Threshold (Physical meaning in Cosine space)
            # 0.15 Cosine Dist ~= 30 degree difference. 
            # This is a reasonable "radius" to consider points connected in embedding space.
            persistence_threshold = 0.15 
            
            # 5. Compute Betti-0 (Number of Connected Components at threshold)
            # H0 Diagram: [0, death_time]
            # If death_time > threshold, the component is still separate at that scale.
            h0_deaths = diagrams[0][:, 1]
            
            # We count how many components are still "alive" (have not merged) at radius 0.15
            # This tells us: "How many distinct semantic clusters exist in this subset?"
            betti_0 = np.sum(h0_deaths > persistence_threshold)
            
            # 6. Compute Betti-1 (Number of Loops/Holes)
            # H1 Diagram: [birth, death]
            betti_1 = 0
            if len(diagrams) > 1 and len(diagrams[1]) > 0:
                births = diagrams[1][:, 0]
                deaths = diagrams[1][:, 1]
                
                # A loop exists if it was born BEFORE the threshold 
                # and persists UNTIL AFTER the threshold.
                # (i.e., the loop is "open" at scale 0.15)
                betti_1 = np.sum((births < persistence_threshold) & (deaths > persistence_threshold))
                
            return betti_0, betti_1
            
        except Exception as e:
            print(f"    ! Topology Error: {e}")
            return np.nan, np.nan
            
    @staticmethod
    def compute_boundary_projection_stats(model, boundary_vec):
        """
        NEW: Direct projection through unembedding.
        If entropy is high, the boundary maps to 'nothing specific' (Null Space).
        """
        b = torch.tensor(boundary_vec, device=model.device, dtype=model.dtype)
        with torch.no_grad():
            logits = model.get_output_embeddings()(b)
            probs = F.softmax(logits, dim=0)
            
            # Shannon Entropy
            entropy = -(probs * torch.log(probs + 1e-10)).sum()
            max_prob = probs.max()
            
        return entropy.item(), max_prob.item()

# ============================================================
# 4. Mechanistic Metrics (Updates: Gradient Sign & Correctness)
# ============================================================

def get_model_layers(model):
    if hasattr(model, "model") and hasattr(model.model, "layers"): return model.model.layers
    if hasattr(model, "transformer") and hasattr(model.transformer, "h"): return model.transformer.h
    raise ValueError("Could not locate layers.")

# --- 1. Fisher Information ---
def run_fisher_analysis(model, tokenizer, prompts, layer_idx, boundary_vec, device, epsilon=0.05, batch_size=4):
    boundary_tensor = torch.tensor(boundary_vec, device=device, dtype=torch.float64)
    target_layer = get_model_layers(model)[layer_idx]
    
    total_kl = 0.0; count = 0
    for i in range(0, len(prompts), batch_size):
        batch_txt = prompts[i:i+batch_size]
        inputs = tokenizer(batch_txt, padding=True, truncation=True, return_tensors="pt").to(device)
        with torch.no_grad():
            clean_out = model(**inputs)
            log_probs_clean = F.log_softmax(clean_out.logits[:, -1, :].double(), dim=-1)
            probs_clean = F.softmax(clean_out.logits[:, -1, :].double(), dim=-1)
            
        def perturb_hook(module, args, output):
            if isinstance(output, tuple):
                h = output[0].double()
                h[:, -1, :] += (epsilon * boundary_tensor) 
                return (h.to(model.dtype),) + output[1:]
            else:
                h = output.double()
                h[:, -1, :] += (epsilon * boundary_tensor)
                return h.to(model.dtype)

        handle = target_layer.register_forward_hook(perturb_hook)
        with torch.no_grad():
            pert_out = model(**inputs)
        handle.remove()
        
        log_probs_pert = F.log_softmax(pert_out.logits[:, -1, :].double(), dim=-1)
        probs_pert = F.softmax(pert_out.logits[:, -1, :].double(), dim=-1)
        
        kl_sym = 0.5 * (F.kl_div(log_probs_clean, probs_pert, reduction='batchmean', log_target=False) +
                        F.kl_div(log_probs_pert, probs_clean, reduction='batchmean', log_target=False))
        total_kl += max(0.0, kl_sym.item())
        count += 1
    return (total_kl / max(count, 1)) / (epsilon**2)

# --- 2. Hessian Curvature ---
def run_hessian_curvature_corrected(model, tokenizer, prompts, layer_idx, boundary_vec, device, batch_size=4):
    """
    Curvature of LOSS along boundary direction: b^T H b.
    DEBUG VERSION: Larger Epsilon + Prints to diagnose 0.0 values.
    """
    # Force float64 for the vector to ensure precision during addition
    boundary_tensor = torch.tensor(boundary_vec, device=device, dtype=torch.float64)
    target_layer = get_model_layers(model)[layer_idx]
    
    total_curv = 0.0
    count = 0
    
    # 1. INCREASED EPSILON
    # Hidden states in LLMs have norms around 100-500. 
    # A unit vector perturbation of 0.1 is 0.02%. That's noise.
    # We try 3.0 to force a visible shift.
    EPS = 3.0 
    
    for i in range(0, len(prompts), batch_size):
        batch_txt = prompts[i:i+batch_size]
        inputs = tokenizer(batch_txt, padding=True, truncation=True, return_tensors="pt").to(device)
        
        def get_loss(shift):
            def perturb_hook(module, args, output):
                if isinstance(output, tuple):
                    h = output[0].double() # Cast to double
                    h[:, -1, :] += shift * boundary_tensor 
                    return (h.to(model.dtype),) + output[1:]
                else:
                    h = output.double()
                    h[:, -1, :] += shift * boundary_tensor 
                    return h.to(model.dtype)
            
            handle = target_layer.register_forward_hook(perturb_hook)
            with torch.no_grad():
                outputs = model(**inputs)
                # We track the loss of the PREDICTED token (Confidence)
                # Use float64 for the calculation
                logits = outputs.logits[:, -1, :].double()
                log_probs = F.log_softmax(logits, dim=-1)
                max_log_prob = log_probs.max(dim=-1).values
                loss = -max_log_prob 
            handle.remove()
            return loss # Returns tensor [Batch]
        
        # Calculate
        L_center = get_loss(0.0)
        L_plus = get_loss(EPS)
        L_minus = get_loss(-EPS)
        
        # 2. DEBUG PRINT (Run only for the first batch of the first layer checked)
        if count == 0 and i == 0:
            diff_p = (L_plus - L_center).mean().item()
            diff_m = (L_minus - L_center).mean().item()
            print(f"    [DEBUG L{layer_idx}] L_center: {L_center.mean().item():.6f}")
            print(f"    [DEBUG L{layer_idx}] L_plus:   {L_plus.mean().item():.6f} (Diff: {diff_p:.8f})")
            print(f"    [DEBUG L{layer_idx}] L_minus:  {L_minus.mean().item():.6f} (Diff: {diff_m:.8f})")
            if abs(diff_p) < 1e-9 and abs(diff_m) < 1e-9:
                print("    ⚠️ WARNING: Loss is identical. Epsilon is too small or Direction is dead.")

        # Second derivative approx: (f(x+h) - 2f(x) + f(x-h)) / h^2
        curvature = (L_plus - 2 * L_center + L_minus) / (EPS ** 2)
        total_curv += curvature.mean().item()
        count += 1
    
    return total_curv / max(count, 1)

# --- 3. Jacobian Amplification ---
def run_jacobian_amplification(model, tokenizer, prompts, layer_idx, boundary_vec, device, epsilon=0.5, batch_size=4):
    boundary_tensor = torch.tensor(boundary_vec, device=device, dtype=model.dtype)
    target_layer = get_model_layers(model)[layer_idx]
    total_amp = 0; count = 0
    
    for i in range(0, len(prompts), batch_size):
        batch_txt = prompts[i:i+batch_size]
        inputs = tokenizer(batch_txt, padding=True, truncation=True, return_tensors="pt").to(device)
        
        clean_out = []
        def clean_h(m, a, o): clean_out.append(o[0] if isinstance(o, tuple) else o)
        h1 = target_layer.register_forward_hook(clean_h)
        with torch.no_grad(): model(**inputs)
        h1.remove()
        
        pert_out = []
        def pre_h(m, args):
            h = args[0].clone()
            h[:, -1, :] += (epsilon * boundary_tensor)
            return (h, *args[1:])
        def post_h(m, a, o): pert_out.append(o[0] if isinstance(o, tuple) else o)
        h2 = target_layer.register_forward_pre_hook(pre_h)
        h3 = target_layer.register_forward_hook(post_h)
        with torch.no_grad(): model(**inputs)
        h2.remove(); h3.remove()
        
        if clean_out and pert_out:
            diff = (pert_out[0][:, -1, :] - clean_out[0][:, -1, :]) / epsilon
            ratio = torch.norm(diff, dim=-1).mean().item()
            total_amp += ratio
            count += 1
    return total_amp / max(count, 1)

# --- 4. Gradient Blockage (UPDATED: Signed) ---
def run_gradient_blockage(model, tokenizer, prompts, layer_idx, boundary_vec, device, batch_size=4):
    """
    Check if 'Uncertainty' gradients align with boundary.
    UPDATED: Returns SIGNED alignment.
    Positive: Boundary vector points TOWARD uncertainty.
    Negative: Boundary vector points AWAY (suppresses uncertainty).
    """
    boundary_tensor = torch.tensor(boundary_vec, device=device, dtype=torch.float32)
    target_words = [" I", " not", " unsure", " maybe", " depends", " unknown"]
    target_ids = []
    for w in target_words:
        ids = tokenizer(w, add_special_tokens=False).input_ids
        if ids: target_ids.append(ids[-1])
    
    if not target_ids: return 0.0
    
    target_layer = get_model_layers(model)[layer_idx]
    grads = []
    def bwd_hook(module, grad_in, grad_out):
        grads.append(grad_out[0][:, -1, :].detach().float())

    total_align = 0; count = 0
    handle = target_layer.register_full_backward_hook(bwd_hook)
    
    for i in range(0, len(prompts), batch_size):
        batch_txt = prompts[i:i+batch_size]
        inputs = tokenizer(batch_txt, padding=True, truncation=True, return_tensors="pt").to(device)
        model.zero_grad()
        out = model(**inputs)
        
        # POSITIVE loss: We want to INCREASE the logits of uncertainty tokens
        loss = out.logits[:, -1, target_ids].sum() 
        loss.backward()
        
        if grads:
            g = grads[-1] # [Batch, Hidden]
            # Signed Cosine similarity
            sims = F.cosine_similarity(g, boundary_tensor.unsqueeze(0), dim=1)
            total_align += sims.mean().item() # No abs()
            count += 1
            grads.clear()
            
    handle.remove()
    return total_align / max(count, 1)

# --- 5. Vocab Subspace Alignment ---
def precompute_vocab_subspace(model, k=100):
    with torch.no_grad():
        W_u = model.get_output_embeddings().weight.data.float()
        try:
            U, S, Vh = torch.linalg.svd(W_u, full_matrices=False)
            return Vh[:k, :]
        except:
            return None

def run_boundary_vocab_alignment(boundary_vec, vocab_V):
    if vocab_V is None: return np.nan
    b = torch.tensor(boundary_vec, device=vocab_V.device, dtype=vocab_V.dtype)
    return (torch.mv(vocab_V, b).norm() / (b.norm() + 1e-9)).item()

def decode_boundary_contrastive(model, tokenizer, boundary_vec, top_k=3):
    vec_t = torch.tensor(boundary_vec, device=model.device, dtype=model.dtype)
    with torch.no_grad():
        logits = model.get_output_embeddings()(vec_t)
        probs = F.softmax(logits, dim=0)
        vals, indices = torch.topk(probs, top_k)
    return [tokenizer.decode(i.item()).strip() for i in indices]


# --- 6. Loss Gradient Alignment (New Metric) ---
def run_loss_gradient_alignment(model, tokenizer, prompts, layer_idx, boundary_vec, device, batch_size=4):
    """
    Computes cosine similarity between the gradient of the Top-1 Prediction Loss 
    and the Boundary Vector.
    
    Metric: Cos( Gradient(-log(P_max)), Boundary )
    Interpretation: 
      > 0: Boundary vector points towards HIGHER confidence (minimized loss).
      < 0: Boundary vector points towards LOWER confidence (uncertainty).
    """
    # 1. Setup Boundary Vector
    boundary_tensor = torch.tensor(boundary_vec, device=device, dtype=torch.float32)
    b_norm = torch.norm(boundary_tensor)
    if b_norm < 1e-9: return 0.0
    boundary_tensor = boundary_tensor / b_norm

    total_sim = 0
    count = 0
    
    # 2. Manual Embedding Injection (The "Nuclear Option" to guarantee graph connectivity)
    embed_layer = model.get_input_embeddings()

    with torch.enable_grad():
        for i in range(0, len(prompts), batch_size):
            batch_txt = prompts[i:i+batch_size]
            
            inputs = tokenizer(batch_txt, padding=True, truncation=True, return_tensors="pt").to(device)
            input_ids = inputs.input_ids
            attention_mask = inputs.attention_mask
            
            # A. Create Embeddings & Enable Gradients
            inputs_embeds = embed_layer(input_ids).detach()
            inputs_embeds.requires_grad_(True)

            model.zero_grad()
            
            # B. Forward Pass (No labels passed!)
            outputs = model(
                inputs_embeds=inputs_embeds,
                attention_mask=attention_mask,
                output_hidden_states=True
            )
            
            # C. Select Target Layer
            # hidden_states[0] = Embeddings, [1] = Layer 0
            try:
                target_hidden = outputs.hidden_states[layer_idx + 1]
            except IndexError:
                target_hidden = outputs.hidden_states[-1]
            
            target_hidden.retain_grad()
            
            # D. Define Loss: Self-Cross-Entropy of the Last Token
            # We want the gradient of the model's confidence in its own top prediction.
            logits = outputs.logits # [Batch, Seq, Vocab]
            
            # Select only the logits corresponding to the last position in the sequence
            last_indices = attention_mask.sum(dim=1) - 1
            last_token_logits = logits[torch.arange(logits.size(0)), last_indices, :] # [Batch, Vocab]
            
            # Find what the model *thinks* comes next (Pseudo-label)
            predicted_ids = last_token_logits.argmax(dim=-1)
            
            # Calculate Cross Entropy against its own prediction
            # Minimizing this loss = Maximizing confidence
            loss = F.cross_entropy(last_token_logits, predicted_ids)
            
            # E. Backward
            loss.backward()
            
            # F. Extract & Compute Metric
            if target_hidden.grad is not None:
                # Get gradient at the last token position
                g_full = target_hidden.grad # [Batch, Seq, Hidden]
                g = g_full[torch.arange(g_full.size(0)), last_indices, :].float()

                # Normalize Gradients (avoid div by zero)
                g_norms = torch.norm(g, dim=1, keepdim=True)
                
                # Filter valid gradients
                valid_mask = (g_norms > 1e-9).squeeze()
                if valid_mask.ndim == 0: valid_mask = valid_mask.unsqueeze(0)
                
                if valid_mask.any():
                    g_normed = g / (g_norms + 1e-9)
                    
                    # Cosine Similarity
                    # Note: Gradient points in direction of INCREASING Loss.
                    # Since Loss is -LogLikelihood, Gradient points towards LOWER Confidence.
                    # Usually we want alignment with the "Direction of Improvement" (Negative Gradient).
                    # But standard cosine similarity is just dot product.
                    # Result > 0 means Boundary is aligned with gradient (Loss Increase / Confidence Drop).
                    # Result < 0 means Boundary is aligned with negative gradient (Loss Decrease / Confidence Boost).
                    sims = torch.matmul(g_normed, boundary_tensor.unsqueeze(1)).squeeze()
                    
                    if sims.ndim == 0:
                        total_sim += sims.item()
                        count += 1
                    else:
                        valid_sims = sims[valid_mask]
                        total_sim += valid_sims.sum().item()
                        count += valid_sims.size(0)
            
            # Cleanup
            target_hidden.grad = None
            inputs_embeds.grad = None

    return total_sim / max(count, 1)


# ============================================================
# 5. Main Loop (With New Metrics)
# ============================================================

def analyze_model_full(model_cfg: ModelCfg, exp_cfg: ExpConfig, datasets: Dict):
    print(f"\n{'='*60}")
    print(f"🔬 FULL MECHANISTIC SUITE: {model_cfg.short_name}")
    print(f"{'='*60}")
    
    tokenizer = AutoTokenizer.from_pretrained(model_cfg.model_id, trust_remote_code=model_cfg.trust_remote_code)
    if tokenizer.pad_token is None: tokenizer.pad_token = tokenizer.eos_token
    model = AutoModelForCausalLM.from_pretrained(model_cfg.model_id, torch_dtype=model_cfg.dtype, trust_remote_code=model_cfg.trust_remote_code, device_map="auto")
    model.eval()
    
    vocab_V = None
    if exp_cfg.compute_mechanics:
        vocab_V = precompute_vocab_subspace(model, k=100)
        if vocab_V is not None: vocab_V = vocab_V.to(model.dtype)

    layers = get_model_layers(model)
    n_layers = len(layers)
    layers_to_scan = exp_cfg.layers_to_scan if exp_cfg.layers_to_scan else list(range(n_layers))
    
    # 1. Extraction
    print("  > Step 1: Extraction...")
    layer_storage = {l: {k: [] for k in datasets.keys()} for l in layers_to_scan}
    
    for bucket, texts in datasets.items():
        for i in tqdm(range(0, len(texts), exp_cfg.batch_size), desc=f"    {bucket}", leave=False):
            inputs = tokenizer(texts[i:i+exp_cfg.batch_size], padding=True, truncation=True, max_length=exp_cfg.max_length, return_tensors="pt").to(model.device)
            with torch.no_grad():
                out = model(**inputs, output_hidden_states=True)
                for l in layers_to_scan:
                    reps = MetricEngine.get_last_token_reps(out.hidden_states[l+1], inputs.attention_mask)
                    layer_storage[l][bucket].append(reps.float().cpu().numpy())

    # 2. Computation
    print("  > Step 2: Metrics...")
    results_geo = []
    results_mech = []
    
    prev_boundaries = {} 
    
    # --- UPDATED HEADER: Added 'G_ALIGN' ---
    print(f"\n  {'L#':<4} {'BOUNDARY':<12} {'EVAL_ON':<10} {'B_NORM':<6} {'STABL':<5} {'FISH':<6} {'HESS':<8} {'G_ALIGN':<7} {'ENT(P)':<6}")
    print(f"  {'-'*100}")

    for l in tqdm(layers_to_scan, desc="    Scanning"):
        data_map = {k: np.concatenate(v, axis=0) if v else np.array([]) for k,v in layer_storage[l].items()}
        
        # A. Geometry
        if exp_cfg.compute_geometry:
            for bucket, X in data_map.items():
                if len(X) < 10: continue
                row = {"model": model_cfg.short_name, "layer": l, "relative_depth": l/n_layers, "bucket": bucket}
                row["lid"] = MetricEngine.compute_lid(X, exp_cfg.lid_k)
                row["isotropy"] = MetricEngine.compute_isotropy(X)
                row["entropy"] = MetricEngine.compute_entropy(X)
                results_geo.append(row)

        # B. Mechanics
        contrast_pairs = [
            ("1_Factual", "2_Impossible", "Impossible"),
            ("1_Factual", "3_Hallucination", "Hallucination")
        ]

        for (pos_key, neg_key, pair_name) in contrast_pairs:
            X_pos = data_map.get(pos_key, [])
            X_neg = data_map.get(neg_key, [])
            
            if len(X_pos) > 10 and len(X_neg) > 10:
                boundary_vec, b_norm = MetricEngine.compute_boundary_vector(X_pos, X_neg)
                
                stability = np.nan
                if pair_name in prev_boundaries:
                    stability = np.dot(boundary_vec, prev_boundaries[pair_name])
                prev_boundaries[pair_name] = boundary_vec.copy()

                eval_targets = [
                    (neg_key, pair_name), 
                    (pos_key, "Factual")  
                ]

                for eval_key, eval_name in eval_targets:
                    subset_prompts = datasets[eval_key][:exp_cfg.advanced_subset_size]
                    
                    mech_row = {
                        "model": model_cfg.short_name, 
                        "layer": l, 
                        "relative_depth": l/n_layers,
                        "boundary_type": pair_name,
                        "evaluated_on": eval_name,
                        "boundary_norm": b_norm,
                        "boundary_stability": stability
                    }
                    
                    ent, max_p = MetricEngine.compute_boundary_projection_stats(model, boundary_vec)
                    mech_row["boundary_entropy"] = ent
                    mech_row["boundary_max_prob"] = max_p
                    
                    if exp_cfg.compute_mechanics:
                        mech_row["fisher_info"] = run_fisher_analysis(model, tokenizer, subset_prompts, l, boundary_vec, model.device)
                        mech_row["hessian_curvature"] = run_hessian_curvature_corrected(model, tokenizer, subset_prompts, l, boundary_vec, model.device)
                        mech_row["jac_amplification"] = run_jacobian_amplification(model, tokenizer, subset_prompts, l, boundary_vec, model.device)
                        mech_row["grad_blockage"] = run_gradient_blockage(model, tokenizer, subset_prompts, l, boundary_vec, model.device)
                        
                        # --- NEW: Loss Gradient Alignment ---
                        mech_row["grad_alignment"] = run_loss_gradient_alignment(model, tokenizer, subset_prompts, l, boundary_vec, model.device)
                        
                        if vocab_V is not None:
                            mech_row["vocab_alignment"] = run_boundary_vocab_alignment(boundary_vec, vocab_V)
                    
                    if exp_cfg.compute_topology:
                        X_curr = data_map.get(eval_key, [])
                        b0, b1 = MetricEngine.compute_boundary_topology(X_curr, boundary_vec)
                        mech_row["betti_0"] = b0
                        mech_row["betti_1"] = b1

                    results_mech.append(mech_row)
                    
                    # --- UPDATED PRINT: Added 'grad_align' ---
                    if (l % 4 == 0 or l == layers_to_scan[-1]) and (pair_name == "Impossible" or pair_name == "Hallucination"):
                        bnorm_s = f"{mech_row.get('boundary_norm',0):.2f}"
                        stbl_s = f"{mech_row.get('boundary_stability',0):.2f}"
                        fish_s = f"{mech_row.get('fisher_info',0):.2f}"
                        hess_s = f"{mech_row.get('hessian_curvature',0):.4f}"
                        
                        # Existing grad blockage (targeted) vs New grad alignment (general loss)
                        # We print the NEW alignment here as requested
                        align_s = f"{mech_row.get('grad_alignment',0):.3f}"
                        ent_s = f"{mech_row.get('boundary_entropy',0):.2f}"
                        
                        p_print = pair_name[:10]
                        e_print = eval_name[:10]
                        print(f"  {l:<4} {p_print:<12} {e_print:<10} {bnorm_s:<6} {stbl_s:<5} {fish_s:<6} {hess_s:<8} {align_s:<7} {ent_s:<6}")

    del model, tokenizer
    torch.cuda.empty_cache()
    gc.collect()
    
    return pd.DataFrame(results_geo), pd.DataFrame(results_mech)

if __name__ == "__main__":
    # Config
    cfg = ExpConfig(
        n_samples=1000,  
        batch_size=8,
        advanced_subset_size=64 
    )
    
    # Run
    buckets = get_real_datasets(cfg)
    # model_cfg = ModelCfg("Qwen/Qwen2.5-1.5B", "Qwen-1.5B", trust_remote_code=True)
    model_cfg = ModelCfg("Qwen/Qwen2.5-3B", "qwen2.5-3B", trust_remote_code=True)
    
    df_geo, df_mech = analyze_model_full(model_cfg, cfg, buckets)
    
    df_geo.to_csv("3B_full_geometry.csv", index=False)
    df_mech.to_csv("3B_full_mechanics.csv", index=False)
    print("\nAnalysis Complete. Files saved.")

/home/ubuntu/venvs/llm-geo/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading datasets (Target: 1000 per bucket)...
    > Loaded 500 from factual.csv (Factual)
    > Loaded 1000 from impossibleQ.csv (Impossible)
    > Building Mixed Hallucination Dataset...
      - Added 817 from TruthfulQA


Repo card metadata block was not found. Setting CardData to empty.


      - Added 1000 from PopQA (Low Pop)
    > Final Hallucination size: 1000

🔬 FULL MECHANISTIC SUITE: qwen2.5-3B


`torch_dtype` is deprecated! Use `dtype` instead!
Loading checkpoint shards: 100%|██████████| 2/2 [00:01<00:00,  1.74it/s]


...

    [DEBUG L0] L_center: 1.001511
    [DEBUG L0] L_plus:   1.144418 (Diff: 0.14290711)
    [DEBUG L0] L_minus:  0.963337 (Diff: -0.03817378)
  0    Impossible   Impossible 0.29   nan   5.29   0.0120   -0.003  11.94 
    [DEBUG L0] L_center: 1.545898
    [DEBUG L0] L_plus:   1.553025 (Diff: 0.00712721)
    [DEBUG L0] L_minus:  1.504084 (Diff: -0.04181370)
  0    Impossible   Factual    0.29   nan   1.99   -0.0058  0.009   11.94 
    [DEBUG L0] L_center: 1.821885
    [DEBUG L0] L_plus:   1.516358 (Diff: -0.30552746)
    [DEBUG L0] L_minus:  0.881370 (Diff: -0.94051528)
  0    Hallucinat   Hallucinat 0.44   nan   12.26  0.0041   -0.009  11.94 
    [DEBUG L0] L_center: 1.545898
    [DEBUG L0] L_plus:   1.532037 (Diff: -0.01386125)
    [DEBUG L0] L_minus:  1.447460 (Diff: -0.09843790)


    Scanning:   3%|▎         | 1/36 [00:34<19:59, 34.28s/it]

  0    Hallucinat   Factual    0.44   nan   1.64   -0.0094  -0.004  11.94 
    [DEBUG L1] L_center: 1.001511
    [DEBUG L1] L_plus:   1.021844 (Diff: 0.02033263)
    [DEBUG L1] L_minus:  1.097777 (Diff: 0.09626579)


    Scanning:   3%|▎         | 1/36 [00:38<22:24, 38.41s/it]


KeyboardInterrupt: 

In [2]:
import os
import torch
import numpy as np
import pandas as pd
import torch.nn.functional as F
from dataclasses import dataclass
from typing import Dict, List, Optional
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM
from tqdm import tqdm

# ============================================================
# 1. Configuration
# ============================================================

@dataclass
class ModelCfg:
    model_id: str
    short_name: str
    dtype: torch.dtype = torch.bfloat16
    trust_remote_code: bool = True
    attn_implementation: str = "eager" 

@dataclass
class AnalysisConfig:
    n_samples: int = 200        
    batch_size: int = 4         
    max_length: int = 128
    seed: int = 42
    
    # Feature Flags
    compute_attn_metrics: bool = True   
    compute_act_metrics: bool = True    
    compute_logit_lens: bool = True     

def set_seed(seed: int):
    torch.manual_seed(seed)
    np.random.seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

# ============================================================
# 2. Data Loading (Updated for Custom Datasets)
# ============================================================

def get_csv_questions(filename: str, n: int) -> List[str]:
    """Helper to read a CSV and find the question column."""
    try:
        if not os.path.exists(filename):
            print(f"  ! Warning: {filename} not found.")
            return []
        
        df = pd.read_csv(filename)
        
        # Try to find a column that looks like questions
        col_candidates = [c for c in df.columns if any(x in c.lower() for x in ['question', 'text', 'prompt', 'input'])]
        target_col = col_candidates[0] if col_candidates else df.columns[0]
        
        # Sample and format
        raw_data = df[target_col].astype(str).tolist()
        if len(raw_data) > n:
            raw_data = list(np.random.choice(raw_data, n, replace=False))
        
        return [f"Question: {q}\nAnswer:" for q in raw_data]
    except Exception as e:
        print(f"  ! Error reading {filename}: {e}")
        return []

def get_datasets(cfg: AnalysisConfig) -> Dict[str, List[str]]:
    set_seed(cfg.seed)
    datasets = {}
    print(f"Loading Data (Target N={cfg.n_samples})...")
    
    # --- 1. Factual (Local CSV) ---
    print("  > Loading Factual (factual.csv)...")
    datasets['Factual'] = get_csv_questions("factual.csv", cfg.n_samples)
    
    # --- 2. Impossible (Local CSV) ---
    print("  > Loading Impossible (impossibleQ.csv)...")
    datasets['Impossible'] = get_csv_questions("impossibleQ.csv", cfg.n_samples)
    
    # --- 3. Hallucinations (Mix: 50% TruthfulQA + 50% Low PopQA) ---
    print("  > Loading Hallucinations (Mix TruthfulQA + PopQA)...")
    try:
        half_n = cfg.n_samples // 2
        
        # A. TruthfulQA
        ds_tqa = load_dataset("truthful_qa", "generation", split="validation", trust_remote_code=True)
        # Shuffle to get random samples
        tqa_questions = ds_tqa.shuffle(seed=cfg.seed)['question'][:half_n]
        
        # B. PopQA (Low Popularity)
        ds_pop = load_dataset("akariasai/PopQA", split="test", trust_remote_code=True)
        df_pop = ds_pop.to_pandas()
        # Sort by s_pop ascending (lowest popularity first)
        df_pop = df_pop.sort_values(by="s_pop", ascending=True)
        pop_questions = df_pop['question'].tolist()[:half_n]
        
        # Combine
        raw_hallucinations = tqa_questions + pop_questions
        
        # Shuffle the mix so they aren't sequential blocks
        np.random.shuffle(raw_hallucinations)
        
        datasets['Hallucinations'] = [f"Question: {q}\nAnswer:" for q in raw_hallucinations]
        
    except Exception as e:
        print(f"  ! Error loading Hallucination datasets: {e}")
        datasets['Hallucinations'] = []

    # Final check
    for k, v in datasets.items():
        print(f"    - {k}: {len(v)} samples")
        
    return datasets

# ============================================================
# 3. Universal Module Finder
# ============================================================

def find_layers(model):
    """Automatically find layer list, Attn blocks, and MLP blocks."""
    layers = None
    if hasattr(model, "model") and hasattr(model.model, "layers"): layers = model.model.layers
    elif hasattr(model, "transformer") and hasattr(model.transformer, "h"): layers = model.transformer.h
    
    if layers is None: raise ValueError("Could not find layers.")
    
    sample_layer = layers[0]
    attn_name = None
    mlp_name = None
    
    for name, mod in sample_layer.named_children():
        if 'attn' in name.lower() or 'attention' in name.lower():
            attn_name = name
        if 'mlp' in name.lower() or 'ffn' in name.lower() or 'feed_forward' in name.lower():
            mlp_name = name
            
    print(f"  [Architecture Detected] Layers: {len(layers)} | Attn: {attn_name} | MLP: {mlp_name}")
    return layers, attn_name, mlp_name

# ============================================================
# 4. Analysis Logic (Updated keys)
# ============================================================

class InternalDynamics:
    @staticmethod
    def entropy(probs, dim=-1, epsilon=1e-10):
        return -torch.sum(probs * torch.log(probs + epsilon), dim=dim)

    @staticmethod
    def get_last_token_idx(attention_mask):
        return (attention_mask.sum(dim=1) - 1).clamp(min=0)

def analyze_dynamics(model_cfg: ModelCfg, exp_cfg: AnalysisConfig, datasets: Dict):
    print(f"\n{'='*60}")
    print(f"🧠 INTERNAL DYNAMICS SCAN: {model_cfg.short_name}")
    print(f"{'='*60}")

    tokenizer = AutoTokenizer.from_pretrained(model_cfg.model_id, trust_remote_code=model_cfg.trust_remote_code)
    if tokenizer.pad_token is None: tokenizer.pad_token = tokenizer.eos_token
    
    model = AutoModelForCausalLM.from_pretrained(
        model_cfg.model_id, 
        torch_dtype=model_cfg.dtype, 
        trust_remote_code=model_cfg.trust_remote_code, 
        device_map="auto",
        attn_implementation=model_cfg.attn_implementation 
    )
    model.eval()

    layers, attn_attr, mlp_attr = find_layers(model)
    n_layers = len(layers)
    
    # Updated keys based on new requirements
    target_buckets = [k for k in ['Factual', 'Impossible', 'Hallucinations'] if k in datasets and len(datasets[k]) > 0]

    if not target_buckets:
        print("No valid datasets found to analyze.")
        return pd.DataFrame()

    # --- Pass 1: Mean Activations for Boundaries ---
    mean_storage = {l: {k: None for k in target_buckets} for l in range(n_layers)}
    counts = {l: {k: 0 for k in target_buckets} for l in range(n_layers)}

    print("\n🚀 Pass 1: Calculating Boundary Vectors...")
    
    for bucket in target_buckets:
        prompts = datasets[bucket]
        for i in tqdm(range(0, len(prompts), exp_cfg.batch_size), desc=f"  Mean: {bucket}", leave=False):
            batch_txt = prompts[i:i+exp_cfg.batch_size]
            inputs = tokenizer(batch_txt, padding=True, truncation=True, max_length=exp_cfg.max_length, return_tensors="pt").to(model.device)
            last_idxs = InternalDynamics.get_last_token_idx(inputs.attention_mask)
            
            with torch.no_grad():
                outputs = model(**inputs, output_hidden_states=True)
                
            for l in range(n_layers):
                h = outputs.hidden_states[l+1] 
                h_last = h[torch.arange(h.size(0)), last_idxs]
                
                h_sum = h_last.sum(dim=0).float().cpu()
                cnt = h_last.size(0)
                
                if mean_storage[l][bucket] is None: mean_storage[l][bucket] = h_sum
                else: mean_storage[l][bucket] += h_sum
                counts[l][bucket] += cnt

    # Compute TWO vectors per layer:
    # 1. Impossible Vector (Impossible - Factual)
    # 2. Hallucination Vector (Hallucination - Factual)
    boundary_vectors = {l: {} for l in range(n_layers)}
    
    # We define 'Factual' as the base/anchor
    base_fact = 'Factual'
    
    if base_fact in target_buckets:
        # Check for Impossible
        if 'Impossible' in target_buckets:
            for l in range(n_layers):
                if counts[l][base_fact] > 0 and counts[l]['Impossible'] > 0:
                    mu_f = mean_storage[l][base_fact] / counts[l][base_fact]
                    mu_i = mean_storage[l]['Impossible'] / counts[l]['Impossible']
                    vec = mu_i - mu_f
                    vec = vec / (vec.norm() + 1e-9)
                    boundary_vectors[l]['Impossible'] = vec.to(model.device).to(model.dtype)

        # Check for Hallucination
        if 'Hallucinations' in target_buckets:
            for l in range(n_layers):
                if counts[l][base_fact] > 0 and counts[l]['Hallucinations'] > 0:
                    mu_f = mean_storage[l][base_fact] / counts[l][base_fact]
                    mu_h = mean_storage[l]['Hallucinations'] / counts[l]['Hallucinations']
                    vec = mu_h - mu_f
                    vec = vec / (vec.norm() + 1e-9)
                    boundary_vectors[l]['Hallucination'] = vec.to(model.device).to(model.dtype)

    del mean_storage, counts
    torch.cuda.empty_cache()

    # --- Pass 2: Detailed Metrics ---
    print("\n🚀 Pass 2: Computing Internal Dynamics...")
    results = []
    
    for bucket in target_buckets:
        prompts = datasets[bucket]
        
        for i in tqdm(range(0, len(prompts), exp_cfg.batch_size), desc=f"  Scan: {bucket}"):
            batch_txt = prompts[i:i+exp_cfg.batch_size]
            inputs = tokenizer(batch_txt, padding=True, truncation=True, max_length=exp_cfg.max_length, return_tensors="pt").to(model.device)
            last_idxs = InternalDynamics.get_last_token_idx(inputs.attention_mask)
            bsz = inputs.input_ids.size(0)
            
            batch_activations = {l: {} for l in range(n_layers)}
            hooks = []
            
            def get_submodule_hook(l, type_key):
                def hook(module, args, output):
                    val = output[0] if isinstance(output, tuple) else output
                    val_last = val[torch.arange(val.size(0)), last_idxs]
                    batch_activations[l][type_key] = val_last.detach()
                return hook

            if exp_cfg.compute_act_metrics:
                for l in range(n_layers):
                    if attn_attr:
                        mod = getattr(layers[l], attn_attr)
                        hooks.append(mod.register_forward_hook(get_submodule_hook(l, 'attn')))
                    if mlp_attr:
                        mod = getattr(layers[l], mlp_attr)
                        hooks.append(mod.register_forward_hook(get_submodule_hook(l, 'mlp')))

            with torch.no_grad():
                outputs = model(**inputs, output_attentions=True, output_hidden_states=True)
            
            for h in hooks: h.remove()
            
            for l in range(n_layers):
                res_row = {
                    "model": model_cfg.short_name, 
                    "bucket": bucket,
                    "layer": l,
                    "relative_depth": l / n_layers
                }
                
                # 1. Attention
                if exp_cfg.compute_attn_metrics:
                    att = outputs.attentions[l]
                    batch_entropies = []
                    batch_sink = []
                    for b_idx in range(bsz):
                        L = last_idxs[b_idx]
                        # Handling case where sequence length might be small
                        if L > 0:
                            attn_dist = att[b_idx, :, L, :L+1] 
                            H_head = InternalDynamics.entropy(attn_dist, dim=-1)
                            batch_entropies.append(H_head.mean().item())
                            sink_val = attn_dist[:, 0].mean().item()
                            batch_sink.append(sink_val)
                        else:
                            batch_entropies.append(0.0)
                            batch_sink.append(0.0)

                    res_row["attn_entropy"] = np.mean(batch_entropies)
                    res_row["attn_sink"] = np.mean(batch_sink)

                # 2. Activation Alignment
                if exp_cfg.compute_act_metrics:
                    h_curr = outputs.hidden_states[l+1]
                    h_last = h_curr[torch.arange(bsz), last_idxs]
                    
                    # Iterate over boundaries: 'Impossible', 'Hallucination'
                    for b_name, b_vec in boundary_vectors[l].items():
                        # Projections
                        proj = (h_last @ b_vec)
                        res_row[f"res_proj_{b_name}"] = proj.mean().item()
                        
                        # Component Alignment
                        if 'attn' in batch_activations[l] and 'mlp' in batch_activations[l]:
                            a_out = batch_activations[l]['attn']
                            m_out = batch_activations[l]['mlp']
                            
                            sim_a = F.cosine_similarity(a_out, b_vec.unsqueeze(0)).mean().item()
                            sim_m = F.cosine_similarity(m_out, b_vec.unsqueeze(0)).mean().item()
                            
                            res_row[f"attn_align_{b_name}"] = sim_a
                            res_row[f"mlp_align_{b_name}"] = sim_m
                    
                    res_row["res_norm"] = h_last.norm(dim=-1).mean().item()

                # 3. Logit Lens
                if exp_cfg.compute_logit_lens and (l % 2 == 0 or l == n_layers-1):
                    h_last = outputs.hidden_states[l+1][torch.arange(bsz), last_idxs]
                    logits = model.lm_head(h_last)
                    probs = F.softmax(logits, dim=-1)
                    
                    ent = InternalDynamics.entropy(probs, dim=-1).mean().item()
                    conf = probs.max(dim=-1).values.mean().item()
                    top_id = torch.argmax(logits[0]).item()
                    top_tok = tokenizer.decode(top_id).strip()
                    
                    res_row["logit_entropy"] = ent
                    res_row["logit_conf"] = conf
                    res_row["logit_top"] = top_tok
                
                results.append(res_row)

    # --- Saving ---
    print("\n💾 Saving Results...")
    if not results:
        print("No results computed.")
        return pd.DataFrame()

    df = pd.DataFrame(results)
    
    numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
    keys = ['model', 'layer', 'relative_depth', 'bucket']
    numeric_cols = [c for c in numeric_cols if c not in keys]
    
    df_agg = df.groupby(keys)[numeric_cols].mean().reset_index()
    
    df.to_csv("8B_dynamics_raw.csv", index=False)
    df_agg.to_csv("8B_dynamics_aggregated.csv", index=False)
    print("Saved dynamics_raw.csv and dynamics_aggregated.csv")
    
    return df_agg

if __name__ == "__main__":
    # Ensure you are logged into Hugging Face if PopQA/TruthfulQA are gated (usually they are public)
    # huggingface-cli login
    
    cfg = AnalysisConfig(n_samples=500, batch_size=4)
    
    # Update model ID if necessary
    model_cfg = ModelCfg("Qwen/Qwen3-8B", "qwen3-8B", trust_remote_code=True)
    
    datasets = get_datasets(cfg)
    df = analyze_dynamics(model_cfg, cfg, datasets)
    
    if not df.empty:
        print("\nSample Data (Layer 20 - Alignment):")
        cols = ['bucket', 'res_proj_Impossible', 'res_proj_Hallucination']
        safe_cols = [c for c in cols if c in df.columns]
        print(df[df['layer'] == 20][safe_cols])

`trust_remote_code` is not supported anymore.
Please check that the Hugging Face dataset 'truthful_qa' isn't based on a loading script and remove `trust_remote_code`.
If the dataset is based on a loading script, please ask the dataset author to remove it and convert it to a standard format like Parquet.


Loading Data (Target N=500)...
  > Loading Factual (factual.csv)...
  > Loading Impossible (impossibleQ.csv)...
  > Loading Hallucinations (Mix TruthfulQA + PopQA)...


`trust_remote_code` is not supported anymore.
Please check that the Hugging Face dataset 'akariasai/PopQA' isn't based on a loading script and remove `trust_remote_code`.
If the dataset is based on a loading script, please ask the dataset author to remove it and convert it to a standard format like Parquet.
Repo card metadata block was not found. Setting CardData to empty.


    - Factual: 500 samples
    - Impossible: 500 samples
    - Hallucinations: 500 samples

🧠 INTERNAL DYNAMICS SCAN: qwen3-8B


Loading checkpoint shards: 100%|██████████| 5/5 [00:02<00:00,  1.74it/s]


  [Architecture Detected] Layers: 36 | Attn: post_attention_layernorm | MLP: mlp

🚀 Pass 1: Calculating Boundary Vectors...



🚀 Pass 2: Computing Internal Dynamics...


  Scan: Hallucinations: 100%|██████████| 125/125 [00:19<00:00,  6.48it/s]



💾 Saving Results...
Saved dynamics_raw.csv and dynamics_aggregated.csv

Sample Data (Layer 20 - Alignment):
            bucket  res_proj_Impossible  res_proj_Hallucination
60         Factual           -26.184500                -38.5940
61  Hallucinations           -10.877141                -17.5230
62      Impossible             2.908637                -17.4655


In [ ]:
import os
import gc
import torch
import numpy as np
import pandas as pd
from tqdm import tqdm
from dataclasses import dataclass
from typing import Dict, List, Tuple
from transformers import AutoTokenizer, AutoModelForCausalLM
from datasets import load_dataset

# ============================================================
# 1. Configuration
# ============================================================

@dataclass
class MicroscopeConfig:
    n_samples: int = 200      # Target sample size
    batch_size: int = 4
    max_length: int = 128
    seed: int = 42
    
    # Analysis Settings
    top_k: int = 10           # Save top 10 most suspicious components per layer
    scan_mlps: bool = True    # Find "Hallucination Neurons"
    scan_heads: bool = True   # Find "Confused Heads"

@dataclass
class ModelCfg:
    model_id: str
    short_name: str
    dtype: torch.dtype = torch.bfloat16
    trust_remote_code: bool = True
    attn_implementation: str = "eager" # Required for head access

def set_seed(seed: int):
    torch.manual_seed(seed)
    np.random.seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

# ============================================================
# 2. Online Statistics Class (Welford's Algorithm)
# ============================================================
class OnlineStats:
    def __init__(self, feature_dim, device='cpu'):
        self.n = 0
        self.mean = torch.zeros(feature_dim, device=device, dtype=torch.float32)
        self.M2 = torch.zeros(feature_dim, device=device, dtype=torch.float32)

    def update(self, x):
        # x: [Batch, Dim]
        batch_n = x.size(0)
        if batch_n == 0: return
        
        # Cast to float32 for stability
        x = x.float()
        new_n = self.n + batch_n
        
        # Batch update Welford
        delta = x - self.mean
        self.mean += delta.sum(dim=0) / new_n
        self.M2 += ((x - self.mean) * delta).sum(dim=0)
        self.n = new_n

    def get_stats(self):
        variance = self.M2 / (self.n - 1) if self.n > 1 else torch.zeros_like(self.M2)
        return self.mean, torch.sqrt(variance + 1e-9)

# ============================================================
# 3. Data Loading (Updated)
# ============================================================

def get_csv_questions(filename: str, n: int) -> List[str]:
    """Helper to read a CSV and find the question column."""
    try:
        if not os.path.exists(filename):
            print(f"  ! Warning: {filename} not found.")
            return []
        
        df = pd.read_csv(filename)
        # Find column containing 'question', 'text', or 'prompt'
        col_candidates = [c for c in df.columns if any(x in c.lower() for x in ['question', 'text', 'prompt', 'input'])]
        target_col = col_candidates[0] if col_candidates else df.columns[0]
        
        raw_data = df[target_col].astype(str).tolist()
        if len(raw_data) > n:
            raw_data = list(np.random.choice(raw_data, n, replace=False))
        
        return [f"Question: {q}\nAnswer:" for q in raw_data]
    except Exception as e:
        print(f"  ! Error reading {filename}: {e}")
        return []

def get_datasets_microscope(cfg: MicroscopeConfig) -> Dict[str, List[str]]:
    set_seed(cfg.seed)
    datasets = {}
    print(f"Loading datasets (Target N={cfg.n_samples})...")
    
    # 1. Factual (Local CSV)
    print("  > Loading Factual (factual.csv)...")
    datasets['Factual'] = get_csv_questions("factual.csv", cfg.n_samples)

    # 2. Impossible (Local CSV)
    print("  > Loading Impossible (impossibleQ.csv)...")
    datasets['Impossible'] = get_csv_questions("impossibleQ.csv", cfg.n_samples)

    # 3. Hallucinations (Mix: 50% TruthfulQA + 50% Low PopQA)
    print("  > Loading Hallucinations (Mix TruthfulQA + PopQA)...")
    try:
        half_n = cfg.n_samples // 2
        
        # A. TruthfulQA
        ds_tqa = load_dataset("truthful_qa", "generation", split="validation", trust_remote_code=True)
        tqa_questions = ds_tqa.shuffle(seed=cfg.seed)['question'][:half_n]
        
        # B. PopQA (Low Popularity)
        ds_pop = load_dataset("akariasai/PopQA", split="test", trust_remote_code=True)
        df_pop = ds_pop.to_pandas()
        df_pop = df_pop.sort_values(by="s_pop", ascending=True) # Hardest first
        pop_questions = df_pop['question'].tolist()[:half_n]
        
        # Combine and shuffle
        raw_hallucinations = tqa_questions + pop_questions
        np.random.shuffle(raw_hallucinations)
        
        datasets['Hallucinations'] = [f"Question: {q}\nAnswer:" for q in raw_hallucinations]
        
    except Exception as e:
        print(f"  ! Error loading Hallucination mix: {e}")
        datasets['Hallucinations'] = []

    # Summary
    for k, v in datasets.items():
        print(f"    - {k}: {len(v)} samples")

    return datasets

# ============================================================
# 4. Helpers for Architecture
# ============================================================

def find_submodules(model, layer_idx):
    """Finds the MLP Down-Proj input (Neuron Activations) and Self-Attn module."""
    layers = model.model.layers if hasattr(model, "model") else model.transformer.h
    layer = layers[layer_idx]
    
    mlp_mod = None
    attn_mod = None
    
    # 1. Find MLP 'down_proj' (Input to this is the activation of the up/gate proj)
    # Llama uses: down_proj(act_fn(gate_proj(x)) * up_proj(x))
    # Hooking the input of down_proj gives us the activations of the intermediate neurons.
    for name, mod in layer.named_modules():
        if 'down_proj' in name or ('c_proj' in name and 'mlp' in name):
            mlp_mod = mod
            break
            
    # 2. Find Attention
    for name, mod in layer.named_modules():
        if 'self_attn' in name or 'attention' in name:
            attn_mod = mod
            break
            
    return mlp_mod, attn_mod

# ============================================================
# 5. The Microscope Engine
# ============================================================

def run_microscope(model_cfg: ModelCfg, exp_cfg: MicroscopeConfig):
    print(f"\n{'='*60}")
    print(f"🔬 COMPONENT MICROSCOPE: {model_cfg.short_name}")
    print(f"{'='*60}")
    
    tokenizer = AutoTokenizer.from_pretrained(model_cfg.model_id, trust_remote_code=True)
    if tokenizer.pad_token is None: tokenizer.pad_token = tokenizer.eos_token
    
    model = AutoModelForCausalLM.from_pretrained(
        model_cfg.model_id, 
        torch_dtype=model_cfg.dtype, 
        trust_remote_code=True, 
        device_map="auto",
        attn_implementation=model_cfg.attn_implementation
    )
    model.eval()
    
    layers = model.model.layers if hasattr(model, "model") else model.transformer.h
    n_layers = len(layers)
    
    datasets = get_datasets_microscope(exp_cfg)
    
    # Clean keys
    target_buckets = [k for k in ['Factual', 'Impossible', 'Hallucinations'] if k in datasets and len(datasets[k]) > 0]
    
    if not target_buckets:
        print("No datasets available.")
        return

    # ------------------------------------------------------------
    # Phase 1: Scan Neurons (MLP)
    # ------------------------------------------------------------
    neuron_results = []
    
    if exp_cfg.scan_mlps:
        print("\n🔎 Scanning MLP Neurons...")
        
        for l in tqdm(range(n_layers), desc="  Layers"):
            mlp_mod, _ = find_submodules(model, l)
            if mlp_mod is None: continue
            
            # Prepare stats containers for all buckets
            stats = {k: None for k in target_buckets}
            
            # Hook logic
            def get_activation_hook(bucket):
                def hook(module, args, output):
                    # args[0] is activations (B, Seq, Hidden)
                    act = args[0] 
                    act_last = act[:, -1, :] 
                    
                    if stats[bucket] is None:
                        stats[bucket] = OnlineStats(act_last.shape[-1], device='cpu')
                    
                    stats[bucket].update(act_last.cpu())
                return hook

            # Run Data for ALL buckets
            for bucket_name in target_buckets:
                handle = mlp_mod.register_forward_hook(get_activation_hook(bucket_name))
                prompts = datasets[bucket_name]
                
                for i in range(0, len(prompts), exp_cfg.batch_size):
                    batch = prompts[i:i+exp_cfg.batch_size]
                    inputs = tokenizer(batch, padding=True, truncation=True, return_tensors="pt").to(model.device)
                    with torch.no_grad(): model(**inputs)
                handle.remove()
            
            # --- Analysis: Compare Factual vs Others ---
            base_key = 'Factual'
            if base_key not in stats or stats[base_key] is None: continue
            
            mu_f, sig_f = stats[base_key].get_stats()
            
            # Define comparisons based on available data
            comparisons = []
            if 'Impossible' in stats and stats['Impossible']: 
                comparisons.append(('Impossible', 'Impossible_vs_Factual'))
            if 'Hallucinations' in stats and stats['Hallucinations']: 
                comparisons.append(('Hallucinations', 'Hallucinations_vs_Factual'))
                
            for comp_key, comp_name in comparisons:
                mu_i, sig_i = stats[comp_key].get_stats()
                
                # Fisher Discriminant Score
                # High positive score = Neuron fires much more for the "comp_key" (Imp/Hall) than Factual
                score = (mu_i - mu_f) / (sig_i + sig_f + 1e-6)
                
                vals, indices = torch.topk(score, exp_cfg.top_k)
                
                for k in range(exp_cfg.top_k):
                    neuron_results.append({
                        "layer": l,
                        "type": "neuron_activation",
                        "comparison_group": comp_name, # Distinguish the two types
                        "index": indices[k].item(),
                        "score": vals[k].item(),
                        "mean_deviant": mu_i[indices[k]].item(),
                        "mean_factual": mu_f[indices[k]].item()
                    })

    # ------------------------------------------------------------
    # Phase 2: Scan Heads (Attention)
    # ------------------------------------------------------------
    head_results = []
    
    if exp_cfg.scan_heads:
        print("\n🔎 Scanning Attention Heads...")
        
        for l in tqdm(range(n_layers), desc="  Layers"):
            num_heads = model.config.num_attention_heads
            stats = {k: OnlineStats(num_heads, 'cpu') for k in target_buckets}
            
            for bucket_name in target_buckets:
                prompts = datasets[bucket_name]
                for i in range(0, len(prompts), exp_cfg.batch_size):
                    batch = prompts[i:i+exp_cfg.batch_size]
                    inputs = tokenizer(batch, padding=True, truncation=True, return_tensors="pt").to(model.device)
                    
                    with torch.no_grad(): 
                        out = model(**inputs, output_attentions=True)
                        
                    attn = out.attentions[l] # [B, Head, Seq, Seq]
                    attn_last = attn[:, :, -1, :] 
                    
                    # Entropy (Confusion)
                    entropy = -torch.sum(attn_last * torch.log(attn_last + 1e-9), dim=-1)
                    stats[bucket_name].update(entropy.cpu())

            # --- Analysis ---
            base_key = 'Factual'
            if base_key not in stats or stats[base_key].n == 0: continue
            
            mu_f, sig_f = stats[base_key].get_stats()
            
            comparisons = []
            if 'Impossible' in stats and stats['Impossible'].n > 0: 
                comparisons.append(('Impossible', 'Impossible_vs_Factual'))
            if 'Hallucinations' in stats and stats['Hallucinations'].n > 0: 
                comparisons.append(('Hallucinations', 'Hallucinations_vs_Factual'))
            
            for comp_key, comp_name in comparisons:
                mu_i, sig_i = stats[comp_key].get_stats()
                
                # Difference in Entropy
                # Positive = Deviant prompt causes more confusion than Factual
                diff = mu_i - mu_f
                
                # We care about magnitude (both much more confused OR much more focused/stubborn)
                vals, indices = torch.topk(diff.abs(), exp_cfg.top_k)
                
                for k in range(exp_cfg.top_k):
                    idx = indices[k].item()
                    head_results.append({
                        "layer": l,
                        "type": "head_entropy",
                        "comparison_group": comp_name,
                        "index": idx,
                        "score": diff[idx].item(), # Signed difference
                        "ent_deviant": mu_i[idx].item(),
                        "ent_factual": mu_f[idx].item()
                    })

    # ------------------------------------------------------------
    # Save
    # ------------------------------------------------------------
    print("\n💾 Saving Component Analysis...")
    if not neuron_results and not head_results:
        print("No results to save.")
        return pd.DataFrame()

    df_n = pd.DataFrame(neuron_results)
    df_h = pd.DataFrame(head_results)
    
    df_full = pd.concat([df_n, df_h], ignore_index=True)
    df_full.to_csv("8B_components_microscope.csv", index=False)
    
    # Summary Print
    print("\nTop 5 Distinct Neurons (By Score):")
    if not df_n.empty:
        print(df_n.sort_values("score", ascending=False).head(5)[['layer', 'index', 'comparison_group', 'score']])
    
    print("\nTop 5 Distinct Heads (By Entropy Diff):")
    if not df_h.empty:
        print(df_h.sort_values("score", ascending=False).head(5)[['layer', 'index', 'comparison_group', 'score']])
        
    return df_full

if __name__ == "__main__":
    # Ensure you are logged into Hugging Face if PopQA/TruthfulQA are gated
    # huggingface-cli login
    
    cfg = MicroscopeConfig(n_samples=500) # Adjust samples as needed
    model_cfg = ModelCfg("Qwen/Qwen3-8B", "qwen3-8B", trust_remote_code=True)
    
    run_microscope(model_cfg, cfg)


🔬 COMPONENT MICROSCOPE: qwen3-8B


Loading checkpoint shards: 100%|██████████| 5/5 [00:01<00:00,  3.07it/s]
Some parameters are on the meta device because they were offloaded to the cpu.
`trust_remote_code` is not supported anymore.
Please check that the Hugging Face dataset 'truthful_qa' isn't based on a loading script and remove `trust_remote_code`.
If the dataset is based on a loading script, please ask the dataset author to remove it and convert it to a standard format like Parquet.


Loading datasets (Target N=500)...
  > Loading Factual (factual.csv)...
  > Loading Impossible (impossibleQ.csv)...
  > Loading Hallucinations (Mix TruthfulQA + PopQA)...


`trust_remote_code` is not supported anymore.
Please check that the Hugging Face dataset 'akariasai/PopQA' isn't based on a loading script and remove `trust_remote_code`.
If the dataset is based on a loading script, please ask the dataset author to remove it and convert it to a standard format like Parquet.
Repo card metadata block was not found. Setting CardData to empty.


    - Factual: 500 samples
    - Impossible: 500 samples
    - Hallucinations: 500 samples

🔎 Scanning MLP Neurons...


  Layers:   0%|          | 0/36 [00:00<?, ?it/s]